# NFL Semantic Model Data Agent Evaluation Notebook

Run this notebook in Microsoft Fabric with the `lh_nfl` Lakehouse attached
after the Gold tables and the `NFL Play by Play Model` semantic model are
refreshed.

The notebook:

- recomputes expected answers from Gold SQL tables
- evaluates the `SM Data Agent` Fabric Data Agent against those answers
- writes the evaluation results to Fabric tables through the Data Agent SDK
- displays summary and row-level details for follow-up tuning

This notebook intentionally excludes unsupported question families such as
true play-action, true tight-window tracking, and rushing yards over expected.


In [ ]:
%pip install -U fabric-data-agent-sdk


## Configuration


In [2]:
from datetime import datetime, timezone
import json
import math
import re
import textwrap

import pandas as pd

GOLD_SCHEMA = "gold"
DATA_AGENT_NAME = "NFL Semantic Model Data Agent"
DATA_AGENT_STAGE = "production"  # Use "sandbox" if evaluating unpublished sandbox changes.
WORKSPACE_NAME = None  # Same workspace as this notebook.

EVALUATION_OUTPUT_TABLE = "semantic_model_data_agent_evaluation"
GROUND_TRUTH_TABLE = "semantic_model_data_agent_ground_truth"
WRITE_GROUND_TRUTH_TABLE = True
USE_CUSTOM_CRITIC_PROMPT = False

# Set to a small integer such as 3 for a quick smoke test. Use None for all cases.
MAX_EVALUATION_CASES = None

RUN_UTC = datetime.now(timezone.utc).isoformat()
print(f"Notebook run UTC: {RUN_UTC}")
print(f"Data Agent: {DATA_AGENT_NAME} ({DATA_AGENT_STAGE})")


def q(table_name: str) -> str:
    return f"`{GOLD_SCHEMA}`.`{table_name}`"


def normalize_sql(sql: str) -> str:
    return textwrap.dedent(sql).strip()


def clean_cell_value(value):
    if value is None:
        return None
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value


def spark_df_to_expected_answer(spark_df, max_rows: int = 20) -> str:
    """Convert a small verifier query result into deterministic text for the evaluator."""
    pdf = spark_df.toPandas()
    if pdf.empty:
        return "No rows returned by the verifier query."

    pdf = pdf.head(max_rows).copy()
    for col in pdf.columns:
        pdf[col] = pdf[col].map(clean_cell_value)

    return pdf.to_string(index=False)


def build_expected_answer(case: dict) -> str:
    result = spark.sql(case["sql"])
    table_text = spark_df_to_expected_answer(result, max_rows=case.get("max_rows", 20))
    note = case.get("note")
    if note:
        return f"{note}\n\nExpected result table:\n{table_text}"
    return f"Expected result table:\n{table_text}"


def case(case_id: str, category: str, question: str, sql: str, note: str | None = None, max_rows: int = 20):
    return {
        "case_id": case_id,
        "category": category,
        "question": question,
        "sql": normalize_sql(sql),
        "note": note,
        "max_rows": max_rows,
    }


available_tables = {
    row.tableName
    for row in spark.sql(f"SHOW TABLES IN `{GOLD_SCHEMA}`").collect()
    if not row.isTemporary
}

required_tables = {
    "agg_player_season",
    "agg_team_game",
    "agg_team_season",
    "agg_team_situation",
    "dim_game",
    "dim_player",
    "dim_team",
    "fact_pass_play",
    "fact_play_core",
    "fact_rush_play",
    "fact_special_teams_play",
    "fact_team_play",
}

missing = sorted(required_tables - available_tables)
if missing:
    raise ValueError(f"Missing required Gold tables: {missing}")

print(f"Gold tables available: {', '.join(sorted(available_tables))}")


StatementMeta(, 681d5a90-2f49-42b9-9a5c-c297cd5ec14d, 9, Finished, Available, Finished, False)

Notebook run UTC: 2026-05-17T05:18:28.130118+00:00
Data Agent: NFL Semantic Model Data Agent (production)
Gold tables available: agg_player_season, agg_team_game, agg_team_season, agg_team_situation, dim_game, dim_player, dim_season_week, dim_team, fact_pass_play, fact_penalty, fact_play_core, fact_player_play_role, fact_rush_play, fact_special_teams_play, fact_team_play


## Evaluation Cases


In [3]:
EVALUATION_CASES = [
    case(
        "scout_qb_hit_vs_clean_epa_2025",
        "scout",
        "For QBs in 2025 with at least 100 clean dropbacks and at least 15 QB-hit dropbacks, who had the largest EPA per dropback differential between clean and QB-hit plays?",
        f"""
        WITH by_state AS (
            SELECT
                passer_player_name,
                offense_team_key,
                is_qb_hit,
                COUNT(*) AS dropbacks,
                SUM(epa) AS epa
            FROM {q("fact_pass_play")}
            WHERE season = 2025
              AND season_type = 'REG'
              AND passer_player_key IS NOT NULL
            GROUP BY passer_player_name, offense_team_key, is_qb_hit
        ),
        pivoted AS (
            SELECT
                passer_player_name,
                offense_team_key,
                SUM(CASE WHEN is_qb_hit THEN dropbacks ELSE 0 END) AS hit_dropbacks,
                SUM(CASE WHEN is_qb_hit THEN epa ELSE 0 END) AS hit_epa,
                SUM(CASE WHEN NOT is_qb_hit THEN dropbacks ELSE 0 END) AS clean_dropbacks,
                SUM(CASE WHEN NOT is_qb_hit THEN epa ELSE 0 END) AS clean_epa
            FROM by_state
            GROUP BY passer_player_name, offense_team_key
        )
        SELECT
            passer_player_name,
            offense_team_key,
            hit_dropbacks,
            clean_dropbacks,
            ROUND(hit_epa / hit_dropbacks, 4) AS hit_epa_per_dropback,
            ROUND(clean_epa / clean_dropbacks, 4) AS clean_epa_per_dropback,
            ROUND((clean_epa / clean_dropbacks) - (hit_epa / hit_dropbacks), 4) AS clean_minus_hit_epa_per_dropback
        FROM pivoted
        WHERE hit_dropbacks >= 15
          AND clean_dropbacks >= 100
        ORDER BY clean_minus_hit_epa_per_dropback DESC NULLS LAST
        LIMIT 5
        """,
        note="Uses the nflverse qb_hit flag, not full charted pressure.",
    ),
    case(
        "fan_regular_season_games_2025",
        "fan",
        "How many regular-season NFL games were played in 2025?",
        f"""
        SELECT
            season,
            season_type,
            COUNT(*) AS regular_season_games
        FROM {q("dim_game")}
        WHERE season = 2025
          AND season_type = 'REG'
        GROUP BY season, season_type
        """,
    ),
    case(
        "beat_49ers_sacks_leaders_2025",
        "beat_reporter",
        "How many sacks did the 49ers defense record in the 2025 regular season, and who were their top 3 sack leaders?",
        f"""
        WITH team_total AS (
            SELECT
                team_key,
                SUM(sacks) AS team_sacks
            FROM {q("agg_team_game")}
            WHERE season = 2025
              AND season_type = 'REG'
              AND team_key = 'SF'
            GROUP BY team_key
        ),
        leaders AS (
            SELECT
                player_name,
                team_key,
                sacks_defense
            FROM {q("agg_player_season")}
            WHERE season = 2025
              AND season_type = 'REG'
              AND team_key = 'SF'
              AND sacks_defense > 0
            ORDER BY sacks_defense DESC
            LIMIT 3
        )
        SELECT
            l.player_name,
            l.team_key,
            t.team_sacks,
            l.sacks_defense
        FROM leaders l
        CROSS JOIN team_total t
        ORDER BY l.sacks_defense DESC
        """,
    ),    
    case(
        "fan_top_5_qbs_passing_yards_2025",
        "fan",
        "List the top 5 quarterbacks by passing yards in the 2025 regular season.",
        f"""
        SELECT
            player_name,
            team_key,
            attempts,
            completions,
            passing_yards,
            passing_tds,
            interceptions,
            ROUND(completion_percentage, 3) AS completion_percentage
        FROM {q("agg_player_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND position = 'QB'
          AND passing_yards > 0
        ORDER BY passing_yards DESC
        LIMIT 5
        """,
    ),
    case(
        "fan_best_red_zone_td_rate_2025",
        "fan",
        "Which team had the best red zone touchdown rate in the 2025 regular season?",
        f"""
        SELECT
            team_key,
            red_zone_drives,
            red_zone_touchdown_drives,
            ROUND(red_zone_drive_td_rate, 3) AS red_zone_drive_td_rate
        FROM {q("agg_team_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND red_zone_drives >= 10
        ORDER BY red_zone_drive_td_rate DESC NULLS LAST
        LIMIT 1
        """,
    ),
    case(
        "fan_chiefs_third_down_rate_2025",
        "fan",
        "What was the Chiefs third-down conversion rate in the 2025 regular season?",
        f"""
        SELECT
            team_key,
            third_down_attempts,
            third_down_conversions,
            ROUND(third_down_conversion_rate, 3) AS third_down_conversion_rate
        FROM {q("agg_team_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND team_key = 'KC'
        """,
    ),
    case(
        "fan_top_10_rushers_epa_per_rush_2025",
        "fan",
        "List the top 10 running backs by EPA per rush in the 2025 regular season, minimum 50 carries.",
        f"""
        SELECT
            player_name,
            team_key,
            position,
            rushing_attempts,
            rushing_yards,
            ROUND(rushing_epa, 3) AS rushing_epa,
            ROUND(rushing_epa_per_play, 4) AS rushing_epa_per_play
        FROM {q("agg_player_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND rushing_attempts >= 50
          AND position IN ('RB', 'FB')
        ORDER BY rushing_epa_per_play DESC NULLS LAST
        LIMIT 10
        """,
    ),
    case(
        "fantasy_highest_deep_cpoe_2025",
        "fantasy",
        "Which quarterback had the highest CPOE on throws of 20 or more air yards in the 2025 regular season, minimum 30 such attempts?",
        f"""
        SELECT
            player_name,
            team_key,
            deep_attempts,
            deep_completions,
            ROUND(deep_completion_percentage, 3) AS deep_completion_percentage,
            ROUND(deep_avg_cpoe, 3) AS deep_avg_cpoe
        FROM {q("agg_player_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND position = 'QB'
          AND deep_attempts >= 30
        ORDER BY deep_avg_cpoe DESC NULLS LAST
        LIMIT 1
        """,
    ),
    case(
        "fan_biggest_single_play_wpa_2025",
        "fan",
        "What was the biggest single-play WPA swing in the 2025 regular season?",
        f"""
        SELECT
            season,
            week,
            game_key,
            offense_team_key,
            defense_team_key,
            ROUND(wpa, 4) AS wpa,
            play_description
        FROM {q("fact_play_core")}
        WHERE season = 2025
          AND season_type = 'REG'
        ORDER BY ABS(wpa) DESC NULLS LAST
        LIMIT 1
        """,
    ),
    case(
        "fantasy_top_10_receivers_yards_2025",
        "fantasy",
        "List the top 10 receivers by receiving yards in the 2025 regular season.",
        f"""
        SELECT
            player_name,
            team_key,
            position,
            targets,
            receptions,
            receiving_yards,
            ROUND(yac_per_reception, 2) AS yac_per_reception
        FROM {q("agg_player_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND receiving_yards > 0
        ORDER BY receiving_yards DESC
        LIMIT 10
        """,
    ),
    case(
        "fan_most_rushing_tds_team_2025",
        "fan",
        "Which team had the most rushing touchdowns in the 2025 regular season?",
        f"""
        SELECT
            team_key,
            SUM(rushing_tds) AS rushing_touchdowns,
            SUM(rushing_attempts) AS rushing_attempts,
            SUM(rushing_yards) AS rushing_yards
        FROM {q("agg_player_season")}
        WHERE season = 2025
          AND season_type = 'REG'
        GROUP BY team_key
        ORDER BY rushing_touchdowns DESC NULLS LAST
        LIMIT 1
        """,
    ),
    case(
        "fantasy_top_5_rbs_rushing_yards_tds_2025",
        "fantasy",
        "List the top 5 running backs by rushing yards in the 2025 regular season and include their rushing touchdowns.",
        f"""
        SELECT
            player_name,
            team_key,
            position,
            rushing_attempts,
            rushing_yards,
            rushing_tds
        FROM {q("agg_player_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND position IN ('RB', 'FB')
          AND rushing_attempts > 0
        ORDER BY rushing_yards DESC NULLS LAST
        LIMIT 5
        """,
    ),
    case(
        "fan_mahomes_completion_td_int_2025",
        "fan",
        "What were Patrick Mahomes' completion percentage and TD-to-INT ratio in the 2025 regular season?",
        f"""
        SELECT
            player_name,
            team_key,
            attempts,
            completions,
            passing_tds,
            interceptions,
            ROUND(completion_percentage, 3) AS completion_percentage,
            ROUND(td_to_int_ratio, 3) AS td_to_int_ratio
        FROM {q("agg_player_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND LOWER(player_name) LIKE '%mahomes%'
        ORDER BY passing_yards DESC
        LIMIT 1
        """,
    ),
    
    case(
        "fan_highest_total_points_game_2025",
        "fan",
        "Which 2025 game had the most total points scored? Show the teams, final score, and week.",
        f"""
        SELECT
            season,
            week,
            game_key,
            away_team,
            away_score,
            home_team,
            home_score,
            total_points
        FROM {q("dim_game")}
        WHERE season = 2025
        ORDER BY total_points DESC NULLS LAST
        LIMIT 1
        """,
    ),
    case(
        "fantasy_top_10_qbs_passing_tds_2025",
        "fantasy",
        "List the top 10 QBs by passing touchdowns in the 2025 regular season, with their interception count shown alongside.",
        f"""
        SELECT
            player_name,
            team_key,
            attempts,
            passing_tds,
            interceptions
        FROM {q("agg_player_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND position = 'QB'
          AND attempts > 0
        ORDER BY passing_tds DESC NULLS LAST, player_name ASC
        LIMIT 10
        """,
    ),
    case(
        "fantasy_kicker_most_fg_attempts_2025",
        "fantasy",
        "Which kicker attempted the most field goals in the 2025 regular season, and what was their make rate?",
        f"""
        SELECT
            player_name,
            team_key,
            field_goal_attempts,
            field_goals_made,
            ROUND(field_goal_make_rate, 3) AS field_goal_make_rate
        FROM {q("agg_player_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND field_goal_attempts > 0
        ORDER BY field_goal_attempts DESC
        LIMIT 1
        """,
    ),
    case(
        "beat_lions_home_road_record_2025",
        "beat_reporter",
        "What was the Detroit Lions' record at home versus on the road in the 2025 regular season?",
        f"""
        SELECT
            team_key,
            home_away,
            COUNT(*) AS games,
            SUM(CASE WHEN game_result = 'Win' THEN 1 ELSE 0 END) AS wins,
            SUM(CASE WHEN game_result = 'Loss' THEN 1 ELSE 0 END) AS losses,
            SUM(CASE WHEN game_result = 'Tie' THEN 1 ELSE 0 END) AS ties
        FROM {q("agg_team_game")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND team_key = 'DET'
        GROUP BY team_key, home_away
        ORDER BY home_away
        """,
    ),
    case(
        "coach_defenses_fewest_explosives_2025",
        "coach",
        "Which 5 defenses allowed the fewest explosive plays per game in the 2025 regular season?",
        f"""
        SELECT
            team_key,
            games,
            explosive_plays_allowed,
            ROUND(explosive_plays_allowed_per_game, 3) AS explosive_plays_allowed_per_game
        FROM {q("agg_team_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND games >= 5
        ORDER BY explosive_plays_allowed_per_game ASC NULLS LAST
        LIMIT 5
        """,
    ),
    case(
        "scout_rb_stuff_rate_2025",
        "scout",
        "Which 10 running backs had the highest stuff rate in the 2025 regular season, minimum 50 carries?",
        f"""
        SELECT
            player_name,
            team_key,
            position,
            rushing_attempts,
            stuffed_rushes,
            ROUND(stuff_rate, 3) AS stuff_rate
        FROM {q("agg_player_season")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND position IN ('RB', 'FB')
          AND rushing_attempts >= 50
        ORDER BY stuff_rate DESC NULLS LAST
        LIMIT 10
        """,
    ),
    case(
        "analyst_home_field_advantage_2025",
        "analyst",
        "Quantify 2025 regular-season home-field advantage: home win rate, average home point differential, average home points, and average away points.",
        f"""
        SELECT
            season,
            COUNT(*) AS games,
            ROUND(SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) / COUNT(*), 3) AS home_win_rate,
            ROUND(AVG(home_score - away_score), 2) AS avg_home_point_differential,
            ROUND(AVG(home_score), 2) AS avg_home_points,
            ROUND(AVG(away_score), 2) AS avg_away_points
        FROM {q("dim_game")}
        WHERE season = 2025
          AND season_type = 'REG'
        GROUP BY season
        """,
    ),
    case(
        "scout_rb_explosive_run_rate_2025",
        "scout",
        "Which 10 running backs had the highest explosive run rate in the 2025 regular season, minimum 50 carries?",
        f"""
        SELECT
            rusher_player_name,
            offense_team_key,
            rusher_position,
            SUM(rush_attempt_count) AS carries,
            SUM(explosive_rush_count) AS explosive_runs,
            ROUND(SUM(explosive_rush_count) / SUM(rush_attempt_count), 3) AS explosive_run_rate
        FROM {q("fact_rush_play")}
        WHERE season = 2025
          AND season_type = 'REG'
          AND is_rb_or_fb_carry
        GROUP BY rusher_player_name, offense_team_key, rusher_position
        HAVING SUM(rush_attempt_count) >= 50
        ORDER BY explosive_run_rate DESC NULLS LAST, carries DESC
        LIMIT 10
        """,
    ),
]

if MAX_EVALUATION_CASES is not None:
    EVALUATION_CASES = EVALUATION_CASES[:MAX_EVALUATION_CASES]

print(f"Evaluation cases configured: {len(EVALUATION_CASES)}")
display(pd.DataFrame([{k: c[k] for k in ["case_id", "category", "question"]} for c in EVALUATION_CASES]))


StatementMeta(, 681d5a90-2f49-42b9-9a5c-c297cd5ec14d, 10, Finished, Available, Finished, False)

Evaluation cases configured: 21


SynapseWidget(Synapse.DataFrame, 4294b461-a519-4851-8280-ab341cc3f49e)

## Generate Ground Truth


In [4]:
ground_truth_records = []

for idx, eval_case in enumerate(EVALUATION_CASES, start=1):
    print(f"[{idx}/{len(EVALUATION_CASES)}] Building expected answer: {eval_case['case_id']}")
    expected_answer = build_expected_answer(eval_case)
    ground_truth_records.append(
        {
            "run_utc": RUN_UTC,
            "case_id": eval_case["case_id"],
            "category": eval_case["category"],
            "question": eval_case["question"],
            "expected_answer": expected_answer,
            "verifier_sql": eval_case["sql"],
            "note": eval_case.get("note"),
        }
    )

ground_truth_df = pd.DataFrame(ground_truth_records)
display(ground_truth_df[["case_id", "category", "question", "expected_answer"]])

if WRITE_GROUND_TRUTH_TABLE:
    spark_ground_truth_df = spark.createDataFrame(ground_truth_df)
    spark_ground_truth_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GROUND_TRUTH_TABLE)
    print(f"Wrote ground truth table: {GROUND_TRUTH_TABLE}")


StatementMeta(, 681d5a90-2f49-42b9-9a5c-c297cd5ec14d, 11, Finished, Available, Finished, False)

[1/21] Building expected answer: scout_qb_hit_vs_clean_epa_2025
[2/21] Building expected answer: fan_regular_season_games_2025
[3/21] Building expected answer: beat_49ers_sacks_leaders_2025
[4/21] Building expected answer: fan_top_5_qbs_passing_yards_2025
[5/21] Building expected answer: fan_best_red_zone_td_rate_2025
[6/21] Building expected answer: fan_chiefs_third_down_rate_2025
[7/21] Building expected answer: fan_top_10_rushers_epa_per_rush_2025
[8/21] Building expected answer: fantasy_highest_deep_cpoe_2025
[9/21] Building expected answer: fan_biggest_single_play_wpa_2025
[10/21] Building expected answer: fantasy_top_10_receivers_yards_2025
[11/21] Building expected answer: fan_most_rushing_tds_team_2025
[12/21] Building expected answer: fantasy_top_5_rbs_rushing_yards_tds_2025
[13/21] Building expected answer: fan_mahomes_completion_td_int_2025
[14/21] Building expected answer: fan_highest_total_points_game_2025
[15/21] Building expected answer: fantasy_top_10_qbs_passing_tds_20

SynapseWidget(Synapse.DataFrame, 39076831-6948-4744-9a2a-a51570cb81ed)

Wrote ground truth table: semantic_model_data_agent_ground_truth


## Run Fabric Data Agent Evaluation


In [5]:
from fabric.dataagent.evaluation import evaluate_data_agent

evaluation_input_df = ground_truth_df[["question", "expected_answer"]].copy()

critic_prompt = """
You are evaluating a Fabric Data Agent answer for an NFL analytics semantic model.

Compare the actual answer to the expected answer. The expected answer is a verifier
SQL result table generated from the Gold layer. Mark the answer as correct only if
the teams, players, rankings, counts, and numeric metrics materially match the
expected result for the user's question.

Rules:
- Formatting differences are acceptable.
- Team abbreviations and full team names are equivalent when unambiguous.
- Numeric rates may differ by small rounding only; counts and rankings should match.
- If the expected answer says a proxy is used, the actual answer should not claim it
  is a richer tracking metric.
- If the actual answer omits a specifically requested column or metric, mark it false
  unless the missing value is clearly inferable from the text.
- If the actual answer gives a different season, season type, player, team, ranking,
  or unsupported metric, mark it false.

Query:
{query}

Expected answer:
{expected_answer}

Actual answer:
{actual_answer}

Is the actual answer equivalent to the expected answer? Answer yes or no, with a
brief reason.
"""

evaluation_kwargs = {
    "workspace_name": WORKSPACE_NAME,
    "table_name": EVALUATION_OUTPUT_TABLE,
    "data_agent_stage": DATA_AGENT_STAGE,
}

if USE_CUSTOM_CRITIC_PROMPT:
    evaluation_kwargs["critic_prompt"] = critic_prompt

evaluation_id = evaluate_data_agent(
    evaluation_input_df,
    DATA_AGENT_NAME,
    **evaluation_kwargs,
)

print(f"Evaluation ID: {evaluation_id}")
print(f"Evaluation output table: {EVALUATION_OUTPUT_TABLE}")
print(f"Evaluation step table: {EVALUATION_OUTPUT_TABLE}_steps")


StatementMeta(, 681d5a90-2f49-42b9-9a5c-c297cd5ec14d, 12, Finished, Available, Finished, False)

**🔗The result does not match the provided ground truth(s):**
- [thread_DRyoAjidKjlFdxT6y0gPMUzl](https://app.fabric.microsoft.com/groups/e22b6dab-2918-468a-98f6-b8435e2c199e/aiskills/dce6b887-cd41-44e5-9cf1-dd2cf8c1d7c2/stage/published/threads/thread_DRyoAjidKjlFdxT6y0gPMUzl/runs/run_fabRCFydmM6ND5nZOIerynAN0FvD/question/msg_gZNRzu3errtOwkZgxi1IGYlr/source/any?debug.dataAgentDeepLinks=1)
- [thread_AquDdg5qzOoulHbETzKR4usL](https://app.fabric.microsoft.com/groups/e22b6dab-2918-468a-98f6-b8435e2c199e/aiskills/dce6b887-cd41-44e5-9cf1-dd2cf8c1d7c2/stage/published/threads/thread_AquDdg5qzOoulHbETzKR4usL/runs/run_fabb0P79jVQW7T3ZuCZhMwD5TNKb/question/msg_eIwNWAysevERk0Qp6TjTEzqw/source/any?debug.dataAgentDeepLinks=1)

100%|██████████| 21/21 [02:10<00:00,  6.20s/it]


Evaluation ID: cec49ea5-9a6d-4bb9-9498-010bf8bf89fb
Evaluation output table: semantic_model_data_agent_evaluation
Evaluation step table: semantic_model_data_agent_evaluation_steps


## Review Results


In [6]:
from fabric.dataagent.evaluation import get_evaluation_details, get_evaluation_summary

summary_df = get_evaluation_summary(EVALUATION_OUTPUT_TABLE, verbose=True)
display(summary_df)

details_df = get_evaluation_details(
    evaluation_id,
    EVALUATION_OUTPUT_TABLE,
    get_all_rows=True,
    verbose=True,
)

display(details_df)

if "evaluation_result" in details_df.columns:
    failures_df = details_df[details_df["evaluation_result"].astype(str).str.lower() != "true"]
    print(f"Non-passing rows: {len(failures_df)}")
    display(failures_df)


StatementMeta(, 681d5a90-2f49-42b9-9a5c-c297cd5ec14d, 13, Finished, Available, Finished, False)

index,evaluation_id,T,F,?,%
0,cec49ea5-9a6d-4bb9-9498-010bf8bf89fb,19,2,0,90.5


SynapseWidget(Synapse.DataFrame, b790d180-caa6-4f3f-a587-02298a2c102a)

index,question,expected_answer,actual_answer,evaluation_judgement,thread_url
0,"For QBs in 2025 with at least 100 clean dropbacks and at least 15 QB-hit dropbacks, who had the largest EPA per dropback differential between clean and QB-hit plays?","Uses the nflverse qb_hit flag, not full charted pressure. Expected result table: passerplayername offenseteamkey hitdropbacks cleandropbacks hitepaperdropback cleanepaperdropback cleanminushitepaper_dropback G.Smith LV 98 405 -1.3678 0.1492 1.5171 M.Mariota WAS 40 204 -1.2261 0.2312 1.4573 T.Shough NO 47 309 -1.2803 0.1624 1.4427 C.Wentz MIN 36 152 -1.1311 0.3099 1.4410 J.Flacco CIN 26 238 -1.2515 0.1727 1.4242","In 2025, among quarterbacks with at least 100 clean dropbacks and at least 15 QB-hit dropbacks, Geno Smith had the largest EPA per dropback differential between clean and QB-hit plays. His differential was +1.52 EPA per dropback (clean: +0.15, QB-hit: –1.37).",True,thread_qkovskpVoAkRLqe9SSzjuCh7
1,How many regular-season NFL games were played in 2025?,Expected result table: season seasontype regularseason_games 2025 REG 272,There were 272 regular-season NFL games played in the 2025 season.,True,thread_xHoIRibhgebFOIRFkdqyq1mm
2,"How many sacks did the 49ers defense record in the 2025 regular season, and who were their top 3 sack leaders?",Expected result table: playername teamkey teamsacks sacksdefense C.Ferrell SF 20 4.0 B.Huff SF 20 4.0 S.Okuayinonu SF 20 3.0,The San Francisco 49ers defense recorded 20 sacks during the 2025 regular season. Their top 3 sack leaders were: B. Huff (4 sacks) C. Ferrell (4 sacks) S. Okuayinonu (3 sacks),True,thread_E3vkZYVDqzZovq6G1IzXBuQ3
3,"List the top 10 QBs by passing touchdowns in the 2025 regular season, with their interception count shown alongside.",Expected result table: playername teamkey attempts passing_tds interceptions M.Stafford LA 617 46 8 J.Goff DET 614 34 8 D.Maye NE 539 31 8 D.Prescott DAL 630 30 10 T.Lawrence JAX 600 29 12 C.Williams CHI 594 27 7 B.Mayfield TB 584 26 11 J.Herbert LAC 566 26 13 B.Nix DEN 638 25 11 J.Allen BUF 506 25 10,"Here are the top 10 quarterbacks by passing touchdowns in the 2025 regular season, along with their interception counts: Matthew Stafford (LA) – 46 TDs, 8 INTs Jared Goff (DET) – 34 TDs, 8 INTs Drake Maye (NE) – 31 TDs, 8 INTs Dak Prescott (DAL) – 30 TDs, 10 INTs Trevor Lawrence (JAX) – 29 TDs, 12 INTs Caleb Williams (CHI) – 27 TDs, 7 INTs Baker Mayfield (TB) – 26 TDs, 11 INTs Justin Herbert (LAC) – 26 TDs, 13 INTs Bo Nix (DEN) – 25 TDs, 11 INTs Josh Allen (BUF) – 25 TDs, 10 INTs",True,thread_dGUjwOKVMJaAqdslMS1tKMTs
4,List the top 10 receivers by receiving yards in the 2025 regular season.,Expected result table: playername teamkey position targets receptions receivingyards yacper_reception J.Smith-Njigba SEA WR 163 119 1793.0 4.44 P.Nacua LA WR 166 129 1715.0 5.16 G.Pickens DAL WR 137 93 1429.0 5.15 J.Chase CIN WR 185 125 1412.0 5.12 A.St. Brown DET WR 172 117 1401.0 4.87 T.McBride ARI TE 170 126 1239.0 4.63 Z.Flowers BAL WR 119 86 1211.0 5.33 C.Olave NO WR 156 100 1163.0 2.89 N.Collins HOU WR 121 71 1117.0 4.56 J.Williams DET WR 102 65 1109.0 6.66,"Here are the top 10 receivers by receiving yards in the 2025 NFL regular season: Jaxon Smith-Njigba (SEA) - 1,793 yards Puka Nacua (LA) - 1,715 yards George Pickens (DAL) - 1,429 yards Ja'Marr Chase (CIN) - 1,412 yards Amon-Ra St. Brown (DET) - 1,401 yards Trey McBride (ARI) - 1,239 yards Zay Flowers (BAL) - 1,211 yards Chris Olave (NO) - 1,163 yards Nico Collins (HOU) - 1,117 yards Jameson Williams (DET) - 1,109 yards",True,thread_8LIhgdk74sNdfoN3aAjxc4uL
5,"List the top 10 running backs by EPA per rush in the 2025 regular season, minimum 50 carries.",Expected result table: playername teamkey position rushingattempts rushingyards rushingepa rushingepaperplay T.Bigsby PHI RB 58 344.0 7.542 0.1300 B.Corum LA RB 145 746.0 15.929 0.1099 R.White TB RB 133 572.0 10.429 0.0784 D.Swift CHI RB 223 1087.0 15.449 0.0693 K.Hunt KC RB 163 611.0 9.669 0.0593

SynapseWidget(Synapse.DataFrame, a0d97ff2-f320-4841-8a85-b885a940b358)

## Tuning Notes

Common follow-ups after a failed row:

- verify whether the Data Agent selected the semantic model data source
- inspect the generated DAX or SQL in the evaluation step table
- add or refine measure descriptions and synonyms in Prep for AI
- hide ambiguous raw columns that invite the agent to bypass curated measures
- add narrowly scoped detail tables only for questions that truly require play-level rows